Detta var ett test jag gjorde när mina chunks såg konstiga ut och jag misstänkte att jag hade problem med massa whitespaces som tog plats. Det visade sig att så var inte fallet.

In [1]:
import os
import json
import re
from pathlib import Path
import pandas as pd


# =========================================================
# 1. Rensa whitespace men behåll stycken
# =========================================================
def clean_text_preserve_paragraphs(text: str) -> str:
    """
    Rensar whitespace utan att förstöra styckeindelning.

    Gör:
    - tar bort spaces/tabs i början och slutet av varje rad
    - ersätter flera spaces/tabs med ett mellanslag
    - behåller stycken
    - normaliserar flera tomrader till max 2 radbrytningar
    """

    # Trimma varje rad
    lines = [line.strip() for line in text.splitlines()]

    cleaned_lines = []
    for line in lines:
        # flera spaces/tabs -> ett mellanslag
        line = re.sub(r"[ \t]+", " ", line)
        cleaned_lines.append(line)

    text = "\n".join(cleaned_lines)

    # ta bort whitespace på tomma rader
    text = re.sub(r"\n\s+\n", "\n\n", text)

    # max två radbrytningar i rad
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# =========================================================
# 2. Hjälpfunktioner för jämförelse
# =========================================================
def count_characters(text: str) -> int:
    return len(text)


def count_paragraphs(text: str) -> int:
    """
    Räknar antal stycken.
    Ett stycke = textblock separerat av tomrad.
    """
    if not text.strip():
        return 0

    paragraphs = re.split(r"\n\s*\n", text.strip())
    paragraphs = [p for p in paragraphs if p.strip()]
    return len(paragraphs)


# =========================================================
# 3. Rensa dataset och spara som cleaned_2_0
# =========================================================
def clean_dataset_and_compare(source_folder: str, target_folder: str) -> pd.DataFrame:
    """
    Läser alla JSON-filer från source_folder,
    rensar data["text"],
    sparar till target_folder med samma mappstruktur,
    och returnerar en DataFrame med jämförelsedata.
    """

    source_folder = Path(source_folder)
    target_folder = Path(target_folder)

    results = []

    for root, _, files in os.walk(source_folder):
        for file in files:
            if not file.endswith(".json"):
                continue

            source_path = Path(root) / file
            relative_path = source_path.relative_to(source_folder)
            target_path = target_folder / relative_path
            target_path.parent.mkdir(parents=True, exist_ok=True)

            with open(source_path, "r", encoding="utf-8") as f:
                data = json.load(f)

            # Om filen saknar textfält, kopiera bara över den
            if "text" not in data or not isinstance(data["text"], str):
                with open(target_path, "w", encoding="utf-8") as f:
                    json.dump(data, f, ensure_ascii=False, indent=2)
                continue

            original_text = data["text"]
            cleaned_text = clean_text_preserve_paragraphs(original_text)

            original_chars = count_characters(original_text)
            cleaned_chars = count_characters(cleaned_text)

            original_paragraphs = count_paragraphs(original_text)
            cleaned_paragraphs = count_paragraphs(cleaned_text)

            chars_removed = original_chars - cleaned_chars
            paragraph_diff = original_paragraphs - cleaned_paragraphs

            # spara ny text
            data["text"] = cleaned_text

            with open(target_path, "w", encoding="utf-8") as f:
                json.dump(data, f, ensure_ascii=False, indent=2)

            results.append({
                "file": str(relative_path),
                "original_chars": original_chars,
                "cleaned_chars": cleaned_chars,
                "chars_removed": chars_removed,
                "original_paragraphs": original_paragraphs,
                "cleaned_paragraphs": cleaned_paragraphs,
                "paragraph_diff": paragraph_diff
            })

    df = pd.DataFrame(results)
    return df


# =========================================================
# 4. Sanity check i kod
# =========================================================
def sanity_check(df: pd.DataFrame):
    print("\n====================")
    print("SANITY CHECK")
    print("====================")

    if df.empty:
        print("Inga JSON-filer med text hittades.")
        return pd.DataFrame(), pd.DataFrame()

    print(f"\nAntal filer: {len(df)}")

    print("\n--- TECKEN ---")
    print(f"Totalt före:     {df['original_chars'].sum():,}")
    print(f"Totalt efter:    {df['cleaned_chars'].sum():,}")
    print(f"Totalt borttaget:{df['chars_removed'].sum():,}")

    print("\n--- STYCKEN ---")
    print(f"Totalt före:     {df['original_paragraphs'].sum():,}")
    print(f"Totalt efter:    {df['cleaned_paragraphs'].sum():,}")
    print(f"Skillnad:        {df['paragraph_diff'].sum():,}")

    changed_paragraphs = df[df["paragraph_diff"] != 0].copy()
    extreme_cases = df[df["chars_removed"] > 1000].copy()

    print("\n--- FILER DÄR STYCKEN ÄNDRATS ---")
    print(f"Antal: {len(changed_paragraphs)}")
    if not changed_paragraphs.empty:
        print(
            changed_paragraphs
            .sort_values("paragraph_diff", ascending=False)
            [["file", "original_paragraphs", "cleaned_paragraphs", "paragraph_diff"]]
            .head(10)
            .to_string(index=False)
        )

    print("\n--- FILER DÄR MEST TEXT TAGITS BORT ---")
    print(
        df.sort_values("chars_removed", ascending=False)
        [["file", "original_chars", "cleaned_chars", "chars_removed"]]
        .head(10)
        .to_string(index=False)
    )

    print("\n--- EXTREMA FALL (>1000 TECKEN BORTTAGNA) ---")
    print(f"Antal: {len(extreme_cases)}")
    if not extreme_cases.empty:
        print(
            extreme_cases
            .sort_values("chars_removed", ascending=False)
            [["file", "chars_removed"]]
            .head(10)
            .to_string(index=False)
        )

    return changed_paragraphs, extreme_cases


# =========================================================
# 5. Inspektera en specifik fil före/efter
# =========================================================
def inspect_file(source_folder: str, target_folder: str, relative_path: str, preview_chars: int = 1500):
    """
    Visar början av original och cleaned-versionen av en specifik fil.
    """

    source_path = Path(source_folder) / relative_path
    target_path = Path(target_folder) / relative_path

    with open(source_path, "r", encoding="utf-8") as f:
        original = json.load(f).get("text", "")

    with open(target_path, "r", encoding="utf-8") as f:
        cleaned = json.load(f).get("text", "")

    print("\n" + "=" * 80)
    print("ORIGINAL")
    print("=" * 80)
    print(original[:preview_chars])

    print("\n" + "=" * 80)
    print("CLEANED")
    print("=" * 80)
    print(cleaned[:preview_chars])


# =========================================================
# 6. Kör här
# =========================================================
source_folder = r"C:\YA BI analyst\Kurser\BI25M AI & IoT\Kunskapskontroll 1\AI-IoT-Kunskapskrav-1\data\cleaned"
target_folder = r"C:\YA BI analyst\Kurser\BI25M AI & IoT\Kunskapskontroll 1\AI-IoT-Kunskapskrav-1\data\cleaned_2.0"

df_compare = clean_dataset_and_compare(source_folder, target_folder)
changed_paragraphs, extreme_cases = sanity_check(df_compare)

# Kolla topp 10 filer där mest text togs bort
print("\n--- TOP 10 CHARS REMOVED ---")
print(df_compare.sort_values("chars_removed", ascending=False).head(10).to_string(index=False))

# Kolla filer där styckeantal ändrats
print("\n--- FILER DÄR PARAGRAF-ANTAL ÄNDRATS ---")
paragraph_issues = df_compare[df_compare["paragraph_diff"] != 0]
if paragraph_issues.empty:
    print("Inga filer fick ändrat antal stycken.")
else:
    print(paragraph_issues.to_string(index=False))

# Exempel: inspektera en specifik fil manuellt
# Byt ut sökvägen nedan mot en fil från df_compare["file"]
# inspect_file(source_folder, target_folder, r"kursnamn\lektion1.json")


SANITY CHECK

Antal filer: 349

--- TECKEN ---
Totalt före:     397,632
Totalt efter:    397,632
Totalt borttaget:0

--- STYCKEN ---
Totalt före:     312
Totalt efter:    312
Skillnad:        0

--- FILER DÄR STYCKEN ÄNDRATS ---
Antal: 0

--- FILER DÄR MEST TEXT TAGITS BORT ---
                 file  original_chars  cleaned_chars  chars_removed
1703530\31459054.json             749            749              0
1703530\31459057.json             442            442              0
1703530\31459076.json             931            931              0
1703530\31459077.json             585            585              0
1703530\31460254.json             686            686              0
1703530\31460310.json             566            566              0
1703530\31460350.json             283            283              0
1703530\31460404.json             257            257              0
1823163\60038464.json             786            786              0
1823163\60039047.json            3214   